In [7]:
from pathlib import Path
import sys

project_root = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "src").is_dir())
sys.path.insert(0, str(project_root))

In [8]:
import time
from src.black_scholes import BlackScholesEngine
from src.binomial_tree import BinomialTreeEngine
from src.monte_carlo import MonteCarloEngine

S, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20

# Black Scholes
t0 = time.perf_counter()
bs_price = BlackScholesEngine(S, K, T, r, sigma).price("call")
t_bs = time.perf_counter() - t0

# Binomial Tree
t0 = time.perf_counter()
tree_price = BinomialTreeEngine(S, K, T, r, sigma, N=500).price("call")
t_tree = time.perf_counter() - t0

# Monte Carlo
t0 = time.perf_counter()
mc_price, mc_se = MonteCarloEngine(S, K, T, r, sigma).price(n_sims=100000, antithetic=True)
t_mc = time.perf_counter() - t0

print(f"{'Model':<16} | {'Price':<10} | {'Error (vs BS)':<15} | {'Latency':<10}")
print("-" * 60)
print(f"{'Black-Scholes':<16} | {bs_price:<10.4f} | {'Baseline':<15} | {t_bs*1000:<8.4f} ms")
print(f"{'Binomial (N=500)':<16} | {tree_price:<10.4f} | {abs(tree_price - bs_price):<15.6f} | {t_tree*1000:<8.4f} ms")
print(f"{'Monte Carlo (100k)':<16} | {mc_price:<10.4f} | {abs(mc_price - bs_price):<15.6f} | {t_mc*1000:<8.4f} ms")

Model            | Price      | Error (vs BS)   | Latency   
------------------------------------------------------------
Black-Scholes    | 10.4506    | Baseline        | 1.5517   ms
Binomial (N=500) | 10.4466    | 0.003998        | 10.2181  ms
Monte Carlo (100k) | 10.4529    | 0.002329        | 6.5696   ms


In [ ]:
from src.visualizer import plot_greeks

# Plots Delta, Gamma, Vega, and Theta profiles across spot prices
plot_greeks(K=100.0, T=1.0, r=0.05, sigma=0.20)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.black_scholes import BlackScholesEngine

S0, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20
n_simulations = 100000

# Generate terminal prices
Z = np.random.standard_normal(n_simulations)
ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
payoffs = np.maximum(ST - K, 0.0)

# Closed-form benchmark
bs_price = BlackScholesEngine(S0, K, T, r, sigma).price("call")

# Plot the 3 subplots from the video
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))

# 1. Terminal price distribution
ax1.hist(ST, bins=50, density=True, alpha=0.7, color="#2E86C1")
ax1.axvline(x=S0, color="#E74C3C", linestyle="--", label="Spot")
ax1.axvline(x=K, color="green", linestyle="--", label="Strike")
ax1.set_title("Terminal Stock Price Distribution")
ax1.set_xlabel("Stock Price")
ax1.set_ylabel("Density")
ax1.legend()

# 2. Payoff distribution
ax2.hist(payoffs, bins=50, density=True, alpha=0.7, color="#2E86C1")
ax2.set_title("Option Payoff Distribution")
ax2.set_xlabel("Payoff ($)")
ax2.set_ylabel("Density")

# 3. Convergence curve
sim_steps = np.arange(100, n_simulations + 1, 500)
running_mc = [np.exp(-r * T) * np.mean(payoffs[:n]) for n in sim_steps]

ax3.plot(sim_steps, running_mc, "b-", alpha=0.8, label="Monte Carlo")
ax3.axhline(y=bs_price, color="r", linestyle="--", label="Black-Scholes")
ax3.set_title("Price Convergence")
ax3.set_xlabel("Number of Simulations")
ax3.set_ylabel("Option Price ($)")
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()